# Challenge 001 — Diagnóstico de Churn
## Análise Exploratória de Dados (EDA)

Carrega os datasets tratados de `data/processed/` e responde:
1. Qual % de contas tiveram churn?
2. Qual % de churn por segmento?
3. Qual % de churn por motivo?

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("data/processed")

accounts = pd.read_parquet(PROCESSED / "accounts.parquet")
churn_events = pd.read_parquet(PROCESSED / "churn_events.parquet")

print(f"Accounts:     {accounts.shape[0]:,} linhas")
print(f"Churn Events: {churn_events.shape[0]:,} linhas")

Accounts:     500 linhas
Churn Events: 600 linhas


### 1. Qual % de contas tiveram churn?

In [2]:
total_contas = accounts.shape[0]

# --- flag_churn da tabela accounts ---
contas_flag = accounts["flag_churn"].sum()

# --- contas distintas que aparecem em churn_events ---
contas_com_evento = churn_events["id_conta"].nunique()
total_eventos = churn_events.shape[0]

print("=" * 55)
print("COMPARAÇÃO: flag_churn (accounts) vs churn_events")
print("=" * 55)
print(f"Total de contas:                         {total_contas}")
print(f"Contas com flag_churn=True (accounts):   {contas_flag}  ({contas_flag/total_contas*100:.1f}%)")
print(f"Contas distintas em churn_events:        {contas_com_evento}  ({contas_com_evento/total_contas*100:.1f}%)")
print(f"Total de eventos de churn:               {total_eventos}")
print(f"Média de eventos por conta com churn:    {total_eventos/contas_com_evento:.1f}")

# Contas em churn_events que NÃO têm flag_churn=True
ids_churn_events = set(churn_events["id_conta"].unique())
ids_flag_true = set(accounts.loc[accounts["flag_churn"], "id_conta"].unique())
so_em_eventos = ids_churn_events - ids_flag_true
so_em_flag = ids_flag_true - ids_churn_events

print(f"\nContas em churn_events SEM flag_churn=True: {len(so_em_eventos)}")
print(f"Contas com flag_churn=True SEM evento:      {len(so_em_flag)}")

# Duplicatas: contas com mais de 1 evento
eventos_por_conta = churn_events.groupby("id_conta").size()
contas_multiplos = (eventos_por_conta > 1).sum()
print(f"\nContas com MÚLTIPLOS eventos de churn:      {contas_multiplos}")
print(f"  Distribuição de eventos por conta:")
display(eventos_por_conta.value_counts().sort_index().rename_axis("eventos").reset_index(name="contas"))

print("\n" + "=" * 55)
print("RESPOSTA FINAL — Usando churn_events (fonte real)")
print("=" * 55)
pct_ja_churn = contas_com_evento / total_contas * 100
pct_nunca_churn = (total_contas - contas_com_evento) / total_contas * 100
print(f"Contas que JÁ tiveram churn:   {contas_com_evento} ({pct_ja_churn:.1f}%)")
print(f"Contas que NUNCA tiveram churn: {total_contas - contas_com_evento} ({pct_nunca_churn:.1f}%)")

Total de contas:       500
Contas com churn:      110
Contas sem churn:      390

% de churn geral:     22.0%


### 2. Qual % de churn por segmento?

Segmentações analisadas: **segmento** (indústria), **plano** e **país**.

In [3]:
def churn_por_grupo(df, coluna):
    resumo = df.groupby(coluna).agg(
        total=("flag_churn", "count"),
        churned=("flag_churn", "sum"),
    )
    resumo["pct_churn"] = (resumo["churned"] / resumo["total"] * 100).round(1)
    return resumo.sort_values("pct_churn", ascending=False)

print("=" * 50)
print("CHURN POR SEGMENTO (INDÚSTRIA)")
print("=" * 50)
display(churn_por_grupo(accounts, "segmento"))

print("\n" + "=" * 50)
print("CHURN POR PLANO")
print("=" * 50)
display(churn_por_grupo(accounts, "plano"))

print("\n" + "=" * 50)
print("CHURN POR PAÍS")
print("=" * 50)
display(churn_por_grupo(accounts, "pais"))

CHURN POR SEGMENTO (INDÚSTRIA)


,total,churned,pct_churn
segmento,,,
DevTools,113,35,31.0
FinTech,112,25,22.3
HealthTech,96,21,21.9
EdTech,79,13,16.5
Cybersecurity,100,16,16.0



CHURN POR PLANO


,total,churned,pct_churn
plano,,,
Enterprise,154,34,22.1
Basic,168,37,22.0
Pro,178,39,21.9



CHURN POR PAÍS


,total,churned,pct_churn
pais,,,
DE,25,8,32.0
US,291,68,23.4
FR,22,5,22.7
IN,49,10,20.4
UK,58,11,19.0
CA,23,4,17.4
AU,32,4,12.5


### 3. Qual % de churn por motivo?

In [4]:
total_eventos = churn_events.shape[0]

motivos = churn_events["motivo"].value_counts()
pct_motivos = (motivos / total_eventos * 100).round(1)

resumo_motivos = pd.DataFrame({
    "qtd": motivos,
    "pct": pct_motivos.map(lambda x: f"{x}%"),
})

print(f"Total de eventos de churn: {total_eventos}")
print()
display(resumo_motivos)

Total de eventos de churn: 600



,qtd,pct
motivo,,
Funcionalidades,114,19.0%
Suporte,104,17.3%
Orçamento,104,17.3%
Desconhecido,95,15.8%
Concorrente,92,15.3%
Preços,91,15.2%
